# iDILI-Predict: UMAP Visualization

Dimensionality reduction and clustering for Cell Painting morphological profiles.

**Pipeline:** StandardScaler -> Harmony batch correction -> PCA -> UMAP -> Leiden clustering

**Requirements:** Install notebook dependencies with `pip install -e ".[notebooks]"`

## 1. Setup

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys
from pathlib import Path

# Add parent directory so we can import idili_predict
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from umap import UMAP
import harmonypy as hm
import scanpy as sc
import anndata as ad

from idili_predict.utils import load_column_types

print("Imports OK")

## 2. Configuration

**Edit these paths to point to your data.** This can be either raw well-level data
or the normalized output from the pipeline (`02_normalized_features.csv`).

In [ ]:
# ========== EDIT THESE PATHS ==========
DATA_FILE = Path("../example_data/63D_well_median_fixed.csv")
DTYPES_FILE = Path("../example_data/column_dtypes.csv")
OUTPUT_DIR = Path("../results/umap_63D")

# ========== COLUMN NAMES ==========
PLATE_COL = "Metadata_PlateID"     # Column for batch correction
COND_COL = "Metadata_COND"         # Column for coloring (PC/NC)
CMPD_COL = "Metadata_CMPD"         # Column for compound labels

# ========== UMAP PARAMETERS ==========
N_PCS = 30              # Number of PCA components
N_NEIGHBORS = 15        # UMAP n_neighbors
MIN_DIST = 0.1          # UMAP min_dist
LEIDEN_RESOLUTION = 1.0 # Leiden clustering resolution

# ========== VALIDATE ==========
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data:   {DATA_FILE} (exists: {DATA_FILE.exists()})")
print(f"Dtypes: {DTYPES_FILE} (exists: {DTYPES_FILE.exists()})")
print(f"Output: {OUTPUT_DIR}")

## 3. Load Data and StandardScaler Normalization

In [ ]:
# Load data
df = pd.read_csv(DATA_FILE, low_memory=False)
print(f"Shape: {df.shape}")

# Remove duplicate columns
dup_cols = [c for c in df.columns if c.endswith(".1")]
if dup_cols:
    df = df.drop(columns=dup_cols)
    print(f"Removed {len(dup_cols)} duplicate columns")

# Load column types
feature_cols_all, metadata_cols_all = load_column_types(DTYPES_FILE)
feature_cols = [c for c in feature_cols_all if c in df.columns
                and pd.api.types.is_numeric_dtype(df[c])]
print(f"Features: {len(feature_cols)}")

In [ ]:
# StandardScaler normalization
X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled feature matrix: {X_scaled.shape}")
print(f"Mean: {X_scaled.mean():.4f}, Std: {X_scaled.std():.4f}")

## 4. Harmony Batch Correction (Optional)

Corrects for plate-level batch effects using the Harmony algorithm.

In [ ]:
if PLATE_COL in df.columns:
    print(f"Running Harmony batch correction on '{PLATE_COL}'...")
    print(f"  Plates: {df[PLATE_COL].nunique()}")
    ho = hm.run_harmony(X_scaled, df, PLATE_COL)
    X_corrected = ho.Z_corr.T
    print(f"  Corrected shape: {X_corrected.shape}")
else:
    print(f"'{PLATE_COL}' not found, skipping batch correction.")
    X_corrected = X_scaled

## 5. PCA

In [ ]:
n_components = min(N_PCS, X_corrected.shape[1])
print(f"Running PCA ({n_components} components)...")

pca = PCA(n_components=n_components, random_state=42)
X_pca = pca.fit_transform(X_corrected)

print(f"PCA output: {X_pca.shape}")
print(f"Cumulative variance (top 10): {pca.explained_variance_ratio_[:10].cumsum()}")

## 6. UMAP Embedding

In [ ]:
print("Running UMAP...")
reducer = UMAP(
    n_neighbors=N_NEIGHBORS,
    min_dist=MIN_DIST,
    n_components=2,
    metric="cosine",
    random_state=42,
    n_jobs=1,
)
embedding = reducer.fit_transform(X_pca)

df["UMAP1"] = embedding[:, 0]
df["UMAP2"] = embedding[:, 1]
print(f"UMAP embedding: {embedding.shape}")

## 7. Leiden Clustering

In [ ]:
# Create AnnData object for scanpy clustering
adata = ad.AnnData(X_pca)
adata.obs_names = [f"well_{i}" for i in range(adata.n_obs)]
adata.obsm["X_umap"] = embedding

# Compute neighbors and cluster
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, use_rep="X")
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION)

df["Leiden_cluster"] = adata.obs["leiden"].values
print(f"Clusters: {df['Leiden_cluster'].nunique()}")
print(df["Leiden_cluster"].value_counts().sort_index())

## 8. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Color by Leiden cluster
scatter1 = axes[0].scatter(
    df["UMAP1"], df["UMAP2"],
    c=df["Leiden_cluster"].astype(int),
    cmap="tab20", s=10, alpha=0.6,
)
axes[0].set_xlabel("UMAP1")
axes[0].set_ylabel("UMAP2")
axes[0].set_title("Leiden Clusters")
plt.colorbar(scatter1, ax=axes[0], label="Cluster")

# Color by condition (PC/NC)
if COND_COL in df.columns:
    cond_colors = df[COND_COL].map({"PC": "red", "NC": "blue"}).fillna("gray")
    for label, color in [("PC (DILI+)", "red"), ("NC (DILI-)", "blue")]:
        cond = "PC" if "PC" in label else "NC"
        mask = df[COND_COL] == cond
        axes[1].scatter(
            df.loc[mask, "UMAP1"], df.loc[mask, "UMAP2"],
            c=color, s=10, alpha=0.4, label=label,
        )
    axes[1].legend()
axes[1].set_xlabel("UMAP1")
axes[1].set_ylabel("UMAP2")
axes[1].set_title("DILI Condition")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "umap_embedding.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to: {OUTPUT_DIR / 'umap_embedding.png'}")

In [ ]:
# Save results
output_file = OUTPUT_DIR / "umap_results.csv"
cols_to_save = [c for c in [PLATE_COL, CMPD_COL, COND_COL,
                            "UMAP1", "UMAP2", "Leiden_cluster"]
                if c in df.columns]
df[cols_to_save].to_csv(output_file, index=False)
print(f"Results saved to: {output_file}")